# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Viditp25/flyrank-intern-/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This notebook completes the ML-07 baseline assignment in five parts:

1. Audit two signals and explain their verdicts.
2. Define one baseline scoring rule with one reason code and one action label.
3. Generate a ranked queue and write it to the required CSV path.
4. Review the top ten recommendations, including why each ranked and what could make the recommendation wrong.
5. Inspect weak picks, check for leakage, and run self-checks.

The generated CSV is saved to `work/outputs/baseline_action_score.csv`. It must remain out of Git because the notebook regenerates it when run.

Run all cells from top to bottom before submitting the notebook.

If using an AI assistant, read `skills/README.md` and load the `building-baselines` and `flyrank/flyrank-data` skills specified by the assignment.

## 1. My Rule and Its Reason Code

Write the rule in plain language, explain the signals behind it, and define one score, one reason code, and one action label.

### Baseline Rule

A page is eligible for human review when both conditions are satisfied:

1. `days_since_last_update >= 91`
2. `impr_days_61_90 >= 100`

For eligible pages, the score equals `impr_days_61_90`. Pages that do not meet both conditions receive a score of zero and are excluded from the ranked queue.

- **Score:** Prior-period impressions for eligible pages
- **Reason code:** `STALE_BUT_VISIBLE`
- **Action label:** `REVIEW_FOR_REFRESH`

The action recommends human review; it does not automatically mean the page should be rewritten.

The scoring rule uses only `days_since_last_update` and `impr_days_61_90`. The decline label, trend direction, recent comparison windows, and product flags are not used to calculate the score.

The 91-day cutoff and 100-impression floor are chosen baseline thresholds for this assignment. Do not describe them as established production thresholds unless repository documentation confirms that claim.

In [19]:
# Section 1: Define the single baseline rule
import numpy as np

STALE_DAYS = 91
VISIBLE_MIN = 100
SCORE_INPUTS = ["days_since_last_update", "impr_days_61_90"]
REASON_CODE = "STALE_BUT_VISIBLE"
ACTION_LABEL = "REVIEW_FOR_REFRESH"


def baseline_rule(frame):
    """Score pages using only the two declared pre-evaluation inputs."""
    out = frame.copy()
    stale = out["days_since_last_update"].ge(STALE_DAYS)
    visible = out["impr_days_61_90"].ge(VISIBLE_MIN)
    eligible = stale & visible

    out["score"] = np.where(eligible, out["impr_days_61_90"], 0)
    out["reason_code"] = np.where(eligible, REASON_CODE, "")
    out["action_label"] = np.where(eligible, ACTION_LABEL, "")
    return out


print("Scoring inputs:", SCORE_INPUTS)
print("Thresholds:", {"stale_days": STALE_DAYS, "visible_min": VISIBLE_MIN})
print("One reason code:", REASON_CODE)
print("One action label:", ACTION_LABEL)


Scoring inputs: ['days_since_last_update', 'impr_days_61_90']
Thresholds: {'stale_days': 91, 'visible_min': 100}
One reason code: STALE_BUT_VISIBLE
One action label: REVIEW_FOR_REFRESH


## 2. Build the Ranked Queue

Apply the baseline rule to the dataset, rank the eligible pages, and write the queue to `work/outputs/baseline_action_score.csv`.

### Ranked Queue and CSV Output

The notebook applies the baseline rule to the dataset and retains only pages with positive scores.

Pages are sorted by score in descending order. When scores are equal, `content_id` is sorted in ascending order to make the ranking deterministic.

The output CSV contains the rank, pseudonymized content identifier, client identifier, score, reason code, action label, update age, and prior-period impressions.

The notebook regenerates `work/outputs/baseline_action_score.csv` on every run. This CSV must remain out of Git, as required by the assignment.

In [20]:
# Section 2: Build the ranked queue and write the CSV

from pathlib import Path
import subprocess

import numpy as np
import pandas as pd

# --------------------------------------------------
# 1. Load and prepare the dataset if df is missing
# --------------------------------------------------

if "df" not in globals():
    DATA_REL = Path("data/raw/content_refresh_anonymized.csv")

    DATA_URL = (
        "https://raw.githubusercontent.com/"
        "Viditp25/flyrank-intern-/main/"
        "data/raw/content_refresh_anonymized.csv"
    )

    # Find the repository root if the repository is available
    def find_repo_root(start=None):
        start = Path(start or Path.cwd()).resolve()

        for path in [start, *start.parents]:
            if (path / DATA_REL).exists():
                return path

        return None

    REPO_ROOT = find_repo_root()

    # Load the dataset
    raw = pd.read_csv(
        REPO_ROOT / DATA_REL if REPO_ROOT else DATA_URL
    )

    df = raw.copy()

    # Derive the historical days 61-90 count from the 90-day total
    # minus the two later 30-day comparison windows.
    source_columns = [
        "content_id", "client_id", "days_since_last_update",
        "impressions_90d", "impressions_last_30d", "impressions_prev_30d",
        "trend_direction",
    ]
    missing_source_columns = [c for c in source_columns if c not in df.columns]
    assert not missing_source_columns, f"Missing required source columns: {missing_source_columns}"

    df["impr_days_61_90"] = (
        df["impressions_90d"]
        - df["impressions_last_30d"]
        - df["impressions_prev_30d"]
    )

    assert df["impr_days_61_90"].notna().all(), "Missing prior-period impression counts detected."
    assert (df["impr_days_61_90"] >= 0).all(), "Negative prior-period impressions detected."

    # Evaluation label only; never used in scoring
    df["is_declining"] = (
        df["trend_direction"] == "down"
    ).astype(int)

    df["trend_defined"] = (
        df["impressions_prev_30d"] > 0
    )

    # Create short client codes for displayed results
    CLIENT_CODE = {
        cid: f"C{i + 1:02d}"
        for i, cid in enumerate(
            df["client_id"].value_counts().index
        )
    }

    df["client_code"] = df["client_id"].map(CLIENT_CODE)

    print("Dataset loaded:", df.shape)

else:
    print("Using the existing dataset in df:", df.shape)


# --------------------------------------------------
# 2. Define rule settings if not already defined
# --------------------------------------------------

STALE_DAYS = globals().get("STALE_DAYS", 91)
VISIBLE_MIN = globals().get("VISIBLE_MIN", 100)

REASON_CODE = globals().get(
    "REASON_CODE", "STALE_BUT_VISIBLE"
)

ACTION_LABEL = globals().get(
    "ACTION_LABEL", "REVIEW_FOR_REFRESH"
)

# --------------------------------------------------
# 3. Define the baseline rule if it is missing
# --------------------------------------------------

if "baseline_rule" not in globals():

    def baseline_rule(frame):
        out = frame.copy()

        stale = (
            out["days_since_last_update"] >= STALE_DAYS
        ).astype(int)

        visible = (
            out["impr_days_61_90"] >= VISIBLE_MIN
        ).astype(int)

        out["score"] = (
            stale
            * visible
            * out["impr_days_61_90"]
        )

        out["reason_code"] = np.where(
            out["score"] > 0,
            REASON_CODE,
            ""
        )

        out["action_label"] = np.where(
            out["score"] > 0,
            ACTION_LABEL,
            ""
        )

        return out


# --------------------------------------------------
# 4. Set the required output path
# --------------------------------------------------

if "OUT_PATH" not in globals():
    DATA_REL = Path("data/raw/content_refresh_anonymized.csv")

    REPO_ROOT = globals().get("REPO_ROOT")

    if REPO_ROOT is None:
        REPO_ROOT = find_repo_root()

    OUT_PATH = (
        (REPO_ROOT or Path.cwd())
        / "work"
        / "outputs"
        / "baseline_action_score.csv"
    )

OUT_REL = "work/outputs/baseline_action_score.csv"


# --------------------------------------------------
# 5. Score, filter, and rank eligible pages
# --------------------------------------------------

required_columns = [
    "content_id",
    "client_id",
    "days_since_last_update",
    "impr_days_61_90",
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

assert not missing_columns, (
    f"Required columns are missing: {missing_columns}"
)

scored = baseline_rule(df)

eligible = scored.loc[
    scored["score"] > 0
].copy()

ranked = (
    eligible
    .sort_values(
        ["score", "content_id"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

ranked["rank"] = ranked.index + 1


# --------------------------------------------------
# 6. Write the ranked queue
# --------------------------------------------------

CSV_COLUMNS = [
    "rank",
    "content_id",
    "client_id",
    "score",
    "reason_code",
    "action_label",
    "days_since_last_update",
    "impr_days_61_90",
]

OUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

ranked[CSV_COLUMNS].to_csv(
    OUT_PATH,
    index=False
)


# --------------------------------------------------
# 7. Display results and validate
# --------------------------------------------------

print(f"Total pages: {len(df):,}")
print(f"Eligible pages: {len(ranked):,}")
print(f"CSV saved to: {OUT_REL}")

display(
    ranked[
        [
            "rank",
            "content_id",
            "client_code",
            "score",
            "reason_code",
            "action_label",
        ]
    ].head(10)
)

assert OUT_PATH.exists(), "CSV was not created."

assert ranked["rank"].tolist() == list(
    range(1, len(ranked) + 1)
), "Rank sequence is invalid."

assert ranked["score"].is_monotonic_decreasing, (
    "Scores are not in descending order."
)

assert ranked["reason_code"].eq(
    REASON_CODE
).all(), "Unexpected reason code."

assert ranked["action_label"].eq(
    ACTION_LABEL
).all(), "Unexpected action label."

print("Ranked queue validation passed.")

Using the existing dataset in df: (30000, 48)
Total pages: 30,000
Eligible pages: 7,506
CSV saved to: work/outputs/baseline_action_score.csv


,rank,content_id,client_code,score,reason_code,action_label
0,1,content_5fe46e04994d,C03,178138,STALE_BUT_VISIBLE,REVIEW_FOR_REFRESH
1,2,content_2dba2b1f9536,C02,165634,STALE_BUT_VISIBLE,REVIEW_FOR_REFRESH
2,3,content_cb112fce36be,C01,112942,STALE_BUT_VISIBLE,REVIEW_FOR_REFRESH
3,4,content_b28d1efd668f,C02,84274,STALE_BUT_VISIBLE,REVIEW_FOR_REFRESH
4,5,content_36ff89c8214e,C01,81700,STALE_BUT_VISIBLE,REVIEW_FOR_REFRESH
5,6,content_2c2606c5d176,C01,79072,STALE_BUT_VISIBLE,REVIEW_FOR_REFRESH
6,7,content_813e88069237,C02,76055,STALE_BUT_VISIBLE,REVIEW_FOR_REFRESH
7,8,content_3d94572c3a35,C01,75252,STALE_BUT_VISIBLE,REVIEW_FOR_REFRESH
8,9,content_124763d39ca5,C02,74850,STALE_BUT_VISIBLE,REVIEW_FOR_REFRESH
9,10,content_396019fec61a,C03,71184,STALE_BUT_VISIBLE,REVIEW_FOR_REFRESH


Ranked queue validation passed.


## 1A. Signal Audits — Inspect Before Interpreting the Score

Both audits evaluate whether selected signals are associated with the observed decline outcome. The outcome is used only for evaluation and is not used by the scoring rule.

Each signal must have a visible bucket table showing the sample count `n`, the number of declining pages, and the observed decline rate.

### Signal A — Staleness / Freshness

The field `days_since_last_update` measures how long a page has gone without an update. This signal is connected to the staleness/freshness concept behind FlyRank's refresh flags.

Compare observed decline rates across update-age buckets. Also compare pages aged 91 days or more with pages updated within 30 days at client level.

### Signal B — Prior Visibility

The field `impr_days_61_90` measures impressions during days 61–90, before the two comparison windows.

Compare observed decline rates across prior-impression buckets. At client level, compare pages with at least 100 prior impressions against pages with 1–99 prior impressions. Keep zero-impression pages separate from this comparison.

### Verdicts

Assign one verdict to each signal: `CONFIRMED`, `OPPOSITE`, `MIXED`, or `FALSE`.

Explain each verdict using the printed bucket rates, sample counts, and client-level comparison. A negative result is acceptable when clearly supported by the evidence.

These audits describe observed associations. They do not establish causation or prove that refreshing a page will improve performance.

In [21]:
# Signal audits: bucket tables with sample counts and explicit verdicts
# The outcome is for evaluation only and is never used by baseline_rule.

import pandas as pd
import numpy as np

# Restrict audit to pages with a measurable decline outcome.
audit_df = df.loc[df["trend_defined"]].copy()
audit_df["is_declining"] = audit_df["is_declining"].astype(int)

# SIGNAL A — staleness/freshness buckets
stale_bins = [-np.inf, 30, 90, 180, np.inf]
stale_labels = ["0-30 days", "31-90 days", "91-180 days", "181+ days"]
audit_df["staleness_bucket"] = pd.cut(
    audit_df["days_since_last_update"], bins=stale_bins,
    labels=stale_labels, right=True, include_lowest=True
)
tbl_a = (
    audit_df.groupby("staleness_bucket", observed=False)
    .agg(n=("is_declining", "size"),
         n_declining=("is_declining", "sum"),
         decline_rate=("is_declining", "mean"))
    .reset_index()
)
tbl_a["decline_rate_pct"] = (100 * tbl_a["decline_rate"]).round(1)
print("SIGNAL A — STALENESS / FRESHNESS (flag-linked)")
display(tbl_a[["staleness_bucket", "n", "n_declining", "decline_rate_pct"]])

# Client-level comparison: 91+ days versus 0-30 days.
audit_df["stale_91_plus"] = audit_df["days_since_last_update"] >= 91
client_stale = audit_df.groupby(["client_id", "stale_91_plus"])["is_declining"].mean().unstack()
if False in client_stale.columns and True in client_stale.columns:
    valid_stale = client_stale.dropna(subset=[False, True])
    stale_gap_pp = (valid_stale[True] - valid_stale[False]) * 100
    stale_clients_higher = int((stale_gap_pp > 0).sum())
    stale_clients_compared = int(len(stale_gap_pp))
    client_sizes = audit_df.groupby("client_id").size().reindex(stale_gap_pp.index)
    stale_weighted_gap_pp = float(np.average(stale_gap_pp, weights=client_sizes)) if stale_clients_compared else np.nan
else:
    stale_clients_higher, stale_clients_compared, stale_weighted_gap_pp = 0, 0, np.nan

# SIGNAL B — prior visibility buckets
visibility_bins = [-np.inf, 0, 99, 999, 9999, np.inf]
visibility_labels = ["0", "1-99", "100-999", "1,000-9,999", "10,000+"]
audit_df["visibility_bucket"] = pd.cut(
    audit_df["impr_days_61_90"], bins=visibility_bins,
    labels=visibility_labels, right=True, include_lowest=True
)
tbl_b = (
    audit_df.groupby("visibility_bucket", observed=False)
    .agg(n=("is_declining", "size"),
         n_declining=("is_declining", "sum"),
         decline_rate=("is_declining", "mean"))
    .reset_index()
)
tbl_b["decline_rate_pct"] = (100 * tbl_b["decline_rate"]).round(1)
print("SIGNAL B — PRIOR VISIBILITY")
display(tbl_b[["visibility_bucket", "n", "n_declining", "decline_rate_pct"]])

# Client-level comparison strictly uses 100+ versus 1-99; zero is excluded.
vis_compare = audit_df.loc[audit_df["impr_days_61_90"] > 0].copy()
vis_compare["visibility_100_plus"] = vis_compare["impr_days_61_90"] >= 100
client_vis = vis_compare.groupby(["client_id", "visibility_100_plus"])["is_declining"].mean().unstack()
if False in client_vis.columns and True in client_vis.columns:
    valid_vis = client_vis.dropna(subset=[False, True])
    vis_gap_pp = (valid_vis[True] - valid_vis[False]) * 100
    vis_clients_higher = int((vis_gap_pp > 0).sum())
    vis_clients_compared = int(len(vis_gap_pp))
else:
    vis_clients_higher, vis_clients_compared = 0, 0

print(
    "Client consistency — staleness: "
    f"{stale_clients_higher}/{stale_clients_compared} clients have a higher decline rate for 91+ day pages "
    f"than for 0-30 day pages; weighted difference={stale_weighted_gap_pp:.1f} percentage points."
    if stale_clients_compared else
    "Client consistency — staleness: insufficient clients have both comparison groups."
)
print(
    "Client consistency — prior visibility: "
    f"{vis_clients_higher}/{vis_clients_compared} clients have a higher decline rate for 100+ impressions "
    "than for 1-99 impressions (zero-impression pages excluded)."
    if vis_clients_compared else
    "Client consistency — prior visibility: insufficient clients have both comparison groups."
)

# Verdicts are transparent, human interpretations of the printed evidence.
# Reassess them if a fresh run changes the bucket tables or client comparisons.
VERDICTS = {
    "A_staleness (days_since_last_update; freshness/staleness flag-linked signal)": "MIXED",
    "B_prior_visibility (impr_days_61_90; visibility floor)": "FALSE",
}
VERDICT_RATIONALE = {
    "A_staleness": (
        "MIXED: the 91-180 day bucket has a higher decline rate than the 0-30 day bucket, "
        "but the 181+ bucket falls again and the direction is not consistent across clients."
    ),
    "B_prior_visibility": (
        "FALSE: decline rates are not consistently increasing across visibility buckets; "
        "the highest-visibility bucket is lower than the middle buckets. Visibility remains an exposure floor, not a demonstrated decline predictor."
    ),
}
ALLOWED_VERDICTS = {"CONFIRMED", "OPPOSITE", "MIXED", "FALSE"}
assert len(VERDICTS) == 2 and set(VERDICTS.values()) <= ALLOWED_VERDICTS

print("VERDICTS AND REASONS")
for key, verdict in VERDICTS.items():
    print(f"{verdict}: {key}")
for rationale in VERDICT_RATIONALE.values():
    print("-", rationale)

assert {"n", "n_declining", "decline_rate"} <= set(tbl_a.columns)
assert {"n", "n_declining", "decline_rate"} <= set(tbl_b.columns)
assert int(tbl_a["n"].sum()) == len(audit_df)
assert int(tbl_b["n"].sum()) == len(audit_df)
print("Both signal tables include n and cover all measurable audit rows.")


SIGNAL A — STALENESS / FRESHNESS (flag-linked)


,staleness_bucket,n,n_declining,decline_rate_pct
0,0-30 days,17454,10473,60.0
1,31-90 days,167,103,61.7
2,91-180 days,8858,5604,63.3
3,181+ days,133,82,61.7


SIGNAL B — PRIOR VISIBILITY


,visibility_bucket,n,n_declining,decline_rate_pct
0,0,959,546,56.9
1,1-99,6426,3931,61.2
2,100-999,9782,5983,61.2
3,"1,000-9,999",8110,5034,62.1
4,"10,000+",1335,768,57.5


Client consistency — staleness: 12/21 clients have a higher decline rate for 91+ day pages than for 0-30 day pages; weighted difference=4.8 percentage points.
Client consistency — prior visibility: 11/28 clients have a higher decline rate for 100+ impressions than for 1-99 impressions (zero-impression pages excluded).
VERDICTS AND REASONS
MIXED: A_staleness (days_since_last_update; freshness/staleness flag-linked signal)
FALSE: B_prior_visibility (impr_days_61_90; visibility floor)
- MIXED: the 91-180 day bucket has a higher decline rate than the 0-30 day bucket, but the 181+ bucket falls again and the direction is not consistent across clients.
- FALSE: decline rates are not consistently increasing across visibility buckets; the highest-visibility bucket is lower than the middle buckets. Visibility remains an exposure floor, not a demonstrated decline predictor.
Both signal tables include n and cover all measurable audit rows.


## 3. Top-10 Review

Review the ten highest-ranked pages. For each page, explain the action, why it ranked, and what evidence could make the recommendation wrong.

### Reviewing the Highest-Ranked Pages

The review uses the ten highest-ranked rows from the baseline queue.

For every row, explain:

1. **Action:** What should happen next.
2. **Why it ranked:** Which baseline conditions were satisfied and why the page received its score.
3. **Confidence note:** What evidence is available or missing.
4. **What could make it wrong:** A specific limitation or contrary piece of evidence.

Trend direction, search volume, average position, and recent impressions may be used for skeptical review only. They must not be used to calculate the score or change the ranking.

The review is decision support, not proof that refreshing a page will improve performance.

In [22]:
# Section 3: Review the top 10 ranked pages

import pandas as pd
import numpy as np

# Select the ten highest-ranked actual queue rows.
top10 = ranked.head(10).copy()
assert len(top10) == min(10, len(ranked)), "Unexpected number of top-ranked rows."

# Ensure optional review fields exist even if the source has no such column.
optional_columns = [
    "avg_position", "ctr", "search_volume", "trend_direction",
    "impressions_last_30d",
]
for col in optional_columns:
    if col not in top10.columns:
        top10[col] = np.nan

# A client-wide concentration of identical update ages may reflect batch/sync behavior.
same_age_share = (
    ranked.groupby("client_id")["days_since_last_update"]
    .agg(lambda s: s.value_counts().iloc[0] / len(s) if len(s) else 0)
)


def explain_rank(r):
    return (
        f"Action={r['action_label']}; reason={r['reason_code']}. "
        f"Last updated {int(r['days_since_last_update'])} days ago; "
        f"days 61-90 impressions={int(r['impr_days_61_90']):,}. "
        f"Both thresholds are met; score equals prior-period impressions."
    )


def explain_risk(r):
    concerns = []
    if pd.isna(r["avg_position"]):
        concerns.append("average search position unavailable")
    elif r["avg_position"] == 0:
        concerns.append("average search position is zero/no position data")
    elif r["avg_position"] > 20:
        concerns.append("average position is beyond the first 20 results")

    if pd.isna(r["search_volume"]) or r["search_volume"] == 0:
        concerns.append("search volume is missing or zero")

    if same_age_share.get(r["client_id"], 0) >= 0.9:
        concerns.append("update age may reflect a client-wide batch or synchronization")

    if pd.notna(r["trend_direction"]) and str(r["trend_direction"]).lower() in {"stable", "up", "upward"}:
        concerns.append("observed trend is stable or upward, not declining")

    if pd.notna(r["impressions_last_30d"]) and r["impr_days_61_90"] > 0:
        if r["impressions_last_30d"] < 0.25 * r["impr_days_61_90"]:
            concerns.append("recent impressions are below 25% of days 61-90 impressions")

    concerns.append("seasonality, search intent, actual edit history, and refresh impact are unknown")
    return "; ".join(concerns)


def confidence_note(r):
    has_concern = (
        pd.isna(r["avg_position"]) or r["avg_position"] == 0
        or pd.isna(r["search_volume"]) or r["search_volume"] == 0
        or (pd.notna(r["avg_position"]) and r["avg_position"] > 20)
        or same_age_share.get(r["client_id"], 0) >= 0.9
        or (pd.notna(r["trend_direction"]) and str(r["trend_direction"]).lower() in {"stable", "up", "upward"})
    )
    return "LOWER: evidence limitations require human review" if has_concern else "PROVISIONAL: rule gates pass; human review is still required"


top10["why_ranked"] = top10.apply(explain_rank, axis=1)
top10["what_could_make_it_wrong"] = top10.apply(explain_risk, axis=1)
top10["confidence_note"] = top10.apply(confidence_note, axis=1)

print(f"TOP-10 REVIEW — {len(top10)} ranked items")
display(top10[[
    "rank", "content_id", "client_code", "action_label", "reason_code", "score",
    "confidence_note", "why_ranked", "what_could_make_it_wrong"
]])

for col in ["action_label", "reason_code", "confidence_note", "why_ranked", "what_could_make_it_wrong"]:
    assert top10[col].notna().all() and top10[col].astype(str).str.strip().ne("").all(), f"Missing review text in {col}"
print("Top-10 review validation passed.")


TOP-10 REVIEW — 10 ranked items


,rank,content_id,client_code,action_label,reason_code,score,confidence_note,why_ranked,what_could_make_it_wrong
0,1,content_5fe46e04994d,C03,REVIEW_FOR_REFRESH,STALE_BUT_VISIBLE,178138,LOWER: evidence limitations require human review,Action=REVIEW_FOR_REFRESH; reason=STALE_BUT_VI...,update age may reflect a client-wide batch or ...
1,2,content_2dba2b1f9536,C02,REVIEW_FOR_REFRESH,STALE_BUT_VISIBLE,165634,LOWER: evidence limitations require human review,Action=REVIEW_FOR_REFRESH; reason=STALE_BUT_VI...,average position is beyond the first 20 result...
2,3,content_cb112fce36be,C01,REVIEW_FOR_REFRESH,STALE_BUT_VISIBLE,112942,LOWER: evidence limitations require human review,Action=REVIEW_FOR_REFRESH; reason=STALE_BUT_VI...,update age may reflect a client-wide batch or ...
3,4,content_b28d1efd668f,C02,REVIEW_FOR_REFRESH,STALE_BUT_VISIBLE,84274,LOWER: evidence limitations require human review,Action=REVIEW_FOR_REFRESH; reason=STALE_BUT_VI...,average position is beyond the first 20 result...
4,5,content_36ff89c8214e,C01,REVIEW_FOR_REFRESH,STALE_BUT_VISIBLE,81700,LOWER: evidence limitations require human review,Action=REVIEW_FOR_REFRESH; reason=STALE_BUT_VI...,search volume is missing or zero; update age m...
5,6,content_2c2606c5d176,C01,REVIEW_FOR_REFRESH,STALE_BUT_VISIBLE,79072,LOWER: evidence limitations require human review,Action=REVIEW_FOR_REFRESH; reason=STALE_BUT_VI...,update age may reflect a client-wide batch or ...
6,7,content_813e88069237,C02,REVIEW_FOR_REFRESH,STALE_BUT_VISIBLE,76055,LOWER: evidence limitations require human review,Action=REVIEW_FOR_REFRESH; reason=STALE_BUT_VI...,average position is beyond the first 20 result...
7,8,content_3d94572c3a35,C01,REVIEW_FOR_REFRESH,STALE_BUT_VISIBLE,75252,LOWER: evidence limitations require human review,Action=REVIEW_FOR_REFRESH; reason=STALE_BUT_VI...,update age may reflect a client-wide batch or ...
8,9,content_124763d39ca5,C02,REVIEW_FOR_REFRESH,STALE_BUT_VISIBLE,74850,LOWER: evidence limitations require human review,Action=REVIEW_FOR_REFRESH; reason=STALE_BUT_VI...,average position is beyond the first 20 result...
9,10,content_396019fec61a,C03,REVIEW_FOR_REFRESH,STALE_BUT_VISIBLE,71184,LOWER: evidence limitations require human review,Action=REVIEW_FOR_REFRESH; reason=STALE_BUT_VI...,update age may reflect a client-wide batch or ...


Top-10 review validation passed.


## 4. Weak Picks and Leakage Check

Inspect whether some highly ranked pages may be poor refresh recommendations. Then verify that the scoring rule does not use evaluation labels, future outcomes, recent comparison windows, or product flags.

### Weak Picks and Leakage Check

A weak pick is a page that meets the baseline thresholds but may not be a useful refresh opportunity. Examples include missing search-position evidence, no measurable search demand, stable or rising trends, sharply reduced recent impressions, or update ages that may reflect a client-wide batch or synchronization.

Weak-pick flags are diagnostic only. They must not change the score or ranking.

The leakage check must confirm that the score uses only `days_since_last_update` and `impr_days_61_90`. The decline label, trend direction, recent comparison windows, future outcomes, and product/page flags must not be scoring inputs.

This baseline identifies pages for human review; it does not establish that refreshing a page causes better performance.

In [23]:

# Section 4: Weak picks + leakage check

import pandas as pd
import numpy as np

# ==================================================
# PART A: WEAK-PICK ANALYSIS
# ==================================================

# Review the top 100 ranked pages without changing their scores.
weak_picks = ranked.head(100).copy()

# Map additional evidence from the original dataset.
# Assumes each content_id identifies one page.
source_df = (
    df.drop_duplicates(subset="content_id")
      .set_index("content_id")
)

optional_columns = [
    "avg_position",
    "ctr",
    "search_volume",
    "trend_direction",
    "impressions_last_30d",
    "impressions_prev_30d",
]

for col in optional_columns:
    if col in source_df.columns and col not in weak_picks.columns:
        weak_picks[col] = weak_picks["content_id"].map(source_df[col])

# Store reasons why a recommendation may need extra review.
weak_picks["weak_pick_reasons"] = ""

# 1. Check average search position.
if "avg_position" in weak_picks.columns:
    missing_position = (
        weak_picks["avg_position"].isna()
        | weak_picks["avg_position"].eq(0)
    )

    weak_picks.loc[
        missing_position, "weak_pick_reasons"
    ] += "Missing or zero average position; "

    weak_picks.loc[
        weak_picks["avg_position"].gt(20),
        "weak_pick_reasons"
    ] += "Average position beyond top 20; "

# 2. Check search-volume evidence.
if "search_volume" in weak_picks.columns:
    missing_volume = (
        weak_picks["search_volume"].isna()
        | weak_picks["search_volume"].eq(0)
    )

    weak_picks.loc[
        missing_volume, "weak_pick_reasons"
    ] += "Missing or zero search volume; "

# 3. Check observed trend.
if "trend_direction" in weak_picks.columns:
    trend = (
        weak_picks["trend_direction"]
        .astype("string")
        .str.lower()
        .str.strip()
    )

    weak_picks.loc[
        trend.isin(["stable", "up", "upward"]),
        "weak_pick_reasons"
    ] += "Trend is stable or upward; "

# 4. Compare recent impressions with days 61-90.
if (
    "impressions_last_30d" in weak_picks.columns
    and "impr_days_61_90" in weak_picks.columns
):
    recent_impressions = weak_picks["impressions_last_30d"]
    older_impressions = weak_picks["impr_days_61_90"]

    weak_picks.loc[
        recent_impressions < 0.25 * older_impressions,
        "weak_pick_reasons"
    ] += "Recent impressions below 25% of days 61-90 impressions; "

# Clean up reason text and flag pages with concerns.
weak_picks["weak_pick_reasons"] = (
    weak_picks["weak_pick_reasons"]
    .str.strip()
    .str.rstrip(";")
)

weak_picks["flagged_as_weak"] = (
    weak_picks["weak_pick_reasons"].ne("")
)

print("WEAK-PICK ANALYSIS")
print("------------------")
print("Ranked pages reviewed:", len(weak_picks))
print(
    "Pages flagged for evidence concerns:",
    int(weak_picks["flagged_as_weak"].sum())
)

display_columns = [
    col for col in [
        "rank",
        "content_id",
        "client_id",
        "score",
        "avg_position",
        "search_volume",
        "trend_direction",
        "weak_pick_reasons",
    ]
    if col in weak_picks.columns
]

display(
    weak_picks.loc[
        weak_picks["flagged_as_weak"],
        display_columns
    ].head(20)
)

# ==================================================
# PART B: LEAKAGE CHECK
# ==================================================

# Evaluation labels, comparison-window variables, future outcomes,
# and known product/page flags must not be score inputs.
#
# Check df.columns below and add any other actual product-flag
# column names present in your dataset.

BANNED = [
    # Outcome and evaluation fields
    "trend_direction",
    "is_declining",
    "trend_defined",
    "decline_rate",

    # Comparison windows and future outcomes
    "impressions_last_30d",
    "impressions_prev_30d",
    "future_performance",
    "future_clicks",
    "future_impressions",
    "future_outcome",

    # Common product/page flag names
    "is_product",
    "is_product_page",
    "product_flag",
]

assert "SCORE_INPUTS" in globals(), (
    "SCORE_INPUTS is missing. Run Section 1 first."
)

leaked_inputs = set(SCORE_INPUTS).intersection(BANNED)

print("\nLEAKAGE CHECK")
print("-------------")
print("Declared score inputs:", SCORE_INPUTS)
print("Forbidden inputs:", BANNED)

assert not leaked_inputs, (
    f"Leakage detected in SCORE_INPUTS: {sorted(leaked_inputs)}"
)

required_score_inputs = {
    "days_since_last_update",
    "impr_days_61_90",
}

assert set(SCORE_INPUTS) == required_score_inputs, (
    "Scoring inputs differ from the declared baseline rule."
)

print("PASS: Declared inputs contain no listed forbidden fields.")
print("PASS: Scoring inputs match the baseline rule.")

# ==================================================
# PART C: VALIDATE THE SCORING FORMULA
# ==================================================

# Recalculate the expected score using the declared rule.
expected_score = (
    (ranked["days_since_last_update"] >= STALE_DAYS).astype(int)
    * (ranked["impr_days_61_90"] >= VISIBLE_MIN).astype(int)
    * ranked["impr_days_61_90"]
)

assert np.allclose(
    ranked["score"].to_numpy(dtype=float),
    expected_score.to_numpy(dtype=float),
    equal_nan=False
), "Score does not match the declared baseline formula."

assert ranked["score"].gt(0).all(), (
    "The ranked queue contains a non-positive score."
)

assert ranked["reason_code"].eq(REASON_CODE).all(), (
    "Unexpected reason code found in ranked queue."
)

assert ranked["action_label"].eq(ACTION_LABEL).all(), (
    "Unexpected action label found in ranked queue."
)

# Verify descending scores and deterministic tie-breaking.
assert ranked["score"].is_monotonic_decreasing, (
    "Ranked scores are not in descending order."
)

expected_order = (
    ranked.sort_values(
        ["score", "content_id"],
        ascending=[False, True]
    )["content_id"]
    .reset_index(drop=True)
)

assert ranked["content_id"].reset_index(drop=True).equals(
    expected_order
), "Ranking or content_id tie-breaking is incorrect."

print("PASS: Scores match the declared baseline formula.")
print("PASS: All ranked rows have the expected reason code and action.")
print("PASS: Scores are descending with deterministic tie-breaking.")
print("PASS: Weak-pick flags do not modify scores or ranking.")


WEAK-PICK ANALYSIS
------------------
Ranked pages reviewed: 100
Pages flagged for evidence concerns: 79


,rank,content_id,client_id,score,avg_position,search_volume,trend_direction,weak_pick_reasons
1,2,content_2dba2b1f9536,client_6208ef0f77,165634,27.9,0.0,stable,Average position beyond top 20; Missing or zer...
3,4,content_b28d1efd668f,client_6208ef0f77,84274,26.2,0.0,stable,Average position beyond top 20; Missing or zer...
4,5,content_36ff89c8214e,client_19581e27de,81700,7.3,0.0,stable,Missing or zero search volume; Trend is stable...
6,7,content_813e88069237,client_6208ef0f77,76055,26.2,0.0,down,Average position beyond top 20; Missing or zer...
8,9,content_124763d39ca5,client_6208ef0f77,74850,33.2,0.0,down,Average position beyond top 20; Missing or zer...
9,10,content_396019fec61a,client_4e07408562,71184,3.8,10.0,stable,Trend is stable or upward; Recent impressions ...
10,11,content_c5063073d048,client_6208ef0f77,70433,12.5,0.0,stable,Missing or zero search volume; Trend is stable...
11,12,content_05e9b4cd9ccf,client_6208ef0f77,69614,22.1,0.0,down,Average position beyond top 20; Missing or zer...
13,14,content_8b36799b7e44,client_6208ef0f77,68432,32.0,0.0,down,Average position beyond top 20; Missing or zer...
16,17,content_3430a8b94511,client_19581e27de,60888,3.3,10.0,stable,Trend is stable or upward



LEAKAGE CHECK
-------------
Declared score inputs: ['days_since_last_update', 'impr_days_61_90']
Forbidden inputs: ['trend_direction', 'is_declining', 'trend_defined', 'decline_rate', 'impressions_last_30d', 'impressions_prev_30d', 'future_performance', 'future_clicks', 'future_impressions', 'future_outcome', 'is_product', 'is_product_page', 'product_flag']
PASS: Declared inputs contain no listed forbidden fields.
PASS: Scoring inputs match the baseline rule.
PASS: Scores match the declared baseline formula.
PASS: All ranked rows have the expected reason code and action.
PASS: Scores are descending with deterministic tie-breaking.
PASS: Weak-pick flags do not modify scores or ranking.


## 5. Self-Check and Submission Checklist

### Assignment Requirements

- [x] Two signal audits include bucket tables and visible sample counts.
- [x] At least one audited signal is linked to the staleness/freshness flag.
- [x] One baseline rule defines one score, one reason code, and one action label.
- [x] The notebook generates the ranked queue at `work/outputs/baseline_action_score.csv`.
- [x] Ten ranked pages are reviewed with an action, ranking rationale, confidence note, and what could make each recommendation wrong.
- [x] Weak-pick analysis and leakage checks are included.



In [24]:
# Self-check: assignment requirements and output integrity
import pandas as pd

checks = []
def check(name, ok, detail=""):
    checks.append({"requirement": name, "pass": bool(ok), "detail": detail})

# 1. Top-10 review contains actual ranked rows.
check("Top-10 review contains up to 10 actual ranked rows", len(top10) == min(10, len(ranked)), f"Reviewed {len(top10)} rows")

# 2-5. Every row has the required review details.
for col, label in [
    ("action_label", "action"),
    ("reason_code", "reason code"),
    ("confidence_note", "confidence note"),
    ("why_ranked", "why it ranked"),
    ("what_could_make_it_wrong", "what could make it wrong"),
]:
    check(
        f"Each top-10 row contains {label}",
        top10[col].notna().all() and top10[col].astype(str).str.strip().ne("").all(),
    )

# 6. CSV exists at the assignment path.
check("CSV exists at the required path", OUT_PATH.exists(), str(OUT_REL))
csv_back = pd.read_csv(OUT_PATH) if OUT_PATH.exists() else pd.DataFrame()

# 7-9. CSV schema, row count, rank sequence, ordering and content match.
check("CSV contains expected columns", not csv_back.empty and set(CSV_COLUMNS).issubset(csv_back.columns), f"CSV rows: {len(csv_back)}")
check(
    "CSV ranks are sequential and scores are descending",
    not csv_back.empty
    and csv_back["rank"].tolist() == list(range(1, len(csv_back) + 1))
    and csv_back["score"].is_monotonic_decreasing,
)
check(
    "CSV identifiers and scores match the ranked queue",
    not csv_back.empty and len(csv_back) == len(ranked)
    and csv_back["content_id"].reset_index(drop=True).equals(ranked["content_id"].reset_index(drop=True))
    and csv_back["score"].reset_index(drop=True).equals(ranked["score"].reset_index(drop=True)),
)

# 10-11. Two signal bucket tables each include n and cover all measurable rows.
check(
    "Signal A staleness table includes n",
    {"n", "n_declining", "decline_rate"} <= set(tbl_a.columns) and int(tbl_a["n"].sum()) == len(audit_df),
    f"Signal A audited rows: {int(tbl_a['n'].sum()):,}",
)
check(
    "Signal B prior-visibility table includes n",
    {"n", "n_declining", "decline_rate"} <= set(tbl_b.columns) and int(tbl_b["n"].sum()) == len(audit_df),
    f"Signal B audited rows: {int(tbl_b['n'].sum()):,}",
)

# 12. Exactly two permitted signal verdicts, including a flag-linked staleness audit.
check(
    "Two signal verdicts use the allowed vocabulary",
    len(VERDICTS) == 2 and set(VERDICTS.values()) <= {"CONFIRMED", "OPPOSITE", "MIXED", "FALSE"},
    str(VERDICTS),
)
check(
    "At least one audit is linked to staleness/freshness",
    any("staleness" in key.lower() or "freshness" in key.lower() for key in VERDICTS),
)

# 13. Declared scoring inputs are exactly the allowed two.
check(
    "Score uses only the two declared inputs",
    set(SCORE_INPUTS) == {"days_since_last_update", "impr_days_61_90"}
    and not set(SCORE_INPUTS).intersection(BANNED),
    str(SCORE_INPUTS),
)

result = pd.DataFrame(checks)
display(result)
if not result["pass"].all():
    display(result.loc[~result["pass"]])
    raise AssertionError("A self-check failed. Fix failed checks before submission.")
print(f"All {len(result)} code-checkable requirements pass.")


,requirement,pass,detail
0,Top-10 review contains up to 10 actual ranked ...,True,Reviewed 10 rows
1,Each top-10 row contains action,True,
2,Each top-10 row contains reason code,True,
3,Each top-10 row contains confidence note,True,
4,Each top-10 row contains why it ranked,True,
5,Each top-10 row contains what could make it wrong,True,
6,CSV exists at the required path,True,work/outputs/baseline_action_score.csv
7,CSV contains expected columns,True,CSV rows: 7506
8,CSV ranks are sequential and scores are descen...,True,
9,CSV identifiers and scores match the ranked queue,True,


All 15 code-checkable requirements pass.
